# 📈 Template: Model Evaluation

**Purpose:** Full evaluation suite for your champion model on the held-out test set.

**Metrics covered:**
- AUC-ROC, Gini Coefficient, KS Statistic
- Precision / Recall / F1 with optimal threshold
- Calibration (Brier Score)
- Gains & Lift analysis
- Confusion matrix
- All-model comparison on test set

**Assumes:** `CHAMPION_NAME`, `CHAMPION_PIPELINE`, `model_registry`, `X_test`, `y_test` from training notebook.


In [ ]:
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, brier_score_loss,
)
from sklearn.calibration import calibration_curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Unseal the test set ───────────────────────────────────────────────────────
print('TEST SET EVALUATION')
print('='*60)
print(f'Champion model : {CHAMPION_NAME}')
print(f'Test rows      : {len(X_test):,}')
print(f'Positive rate  : {y_test.mean():.2%}')
print()

y_test_proba = CHAMPION_PIPELINE.predict_proba(X_test)[:, 1]
y_test_pred  = CHAMPION_PIPELINE.predict(X_test)
test_auc     = roc_auc_score(y_test, y_test_proba)

print(f'Test AUC-ROC : {test_auc:.4f}')
print()
print(classification_report(y_test, y_test_pred))


## 1️⃣  Optimal Threshold (Max F1)

In [ ]:
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_test_proba)
f1_scores        = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh_idx  = np.argmax(f1_scores[:-1])
best_threshold   = thresholds_pr[best_thresh_idx]
best_f1          = f1_scores[best_thresh_idx]

print(f'Optimal threshold (max F1): {best_threshold:.3f}')
print(f'F1 at optimal threshold   : {best_f1:.4f}')
print()

y_test_pred_opt = (y_test_proba >= best_threshold).astype(int)
print(f'Classification report at threshold = {best_threshold:.3f}:')
print(classification_report(y_test, y_test_pred_opt))

# ── Precision-Recall curve ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=100)

ap = average_precision_score(y_test, y_test_proba)
axes[0].plot(recalls[:-1], precisions[:-1], color='#1a434e', lw=2, label=f'AP = {ap:.4f}')
axes[0].axhline(y=y_test.mean(), color='grey', ls='--', lw=1, label=f'Baseline ({y_test.mean():.2%})')
axes[0].scatter(recalls[best_thresh_idx], precisions[best_thresh_idx], s=100,
                color='#e74c3c', zorder=5, label=f'Optimal thr = {best_threshold:.3f}')
axes[0].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[0].legend(fontsize=9)

axes[1].plot(thresholds_pr, precisions[:-1], color='#2980b9', lw=2, label='Precision')
axes[1].plot(thresholds_pr, recalls[:-1],    color='#27ae60', lw=2, label='Recall')
axes[1].plot(thresholds_pr, f1_scores[:-1],  color='#e67e22', lw=2, label='F1')
axes[1].axvline(x=best_threshold, color='#e74c3c', ls='--', lw=1.5, label=f'Optimal = {best_threshold:.3f}')
axes[1].set(xlabel='Threshold', ylabel='Score', title='Precision / Recall / F1 vs Threshold')
axes[1].legend(fontsize=9)

sns.despine(); plt.tight_layout(); plt.show()


## 2️⃣  Gini Coefficient & KS Statistic

In [ ]:
gini = 2 * test_auc - 1
fpr_ks, tpr_ks, thresholds_ks = roc_curve(y_test, y_test_proba)
ks_stat   = float(np.max(tpr_ks - fpr_ks))
ks_thresh = thresholds_ks[np.argmax(tpr_ks - fpr_ks)]

print(f'Gini Coefficient : {gini:.4f}')
print(f'KS Statistic     : {ks_stat:.4f}  (at threshold {ks_thresh:.3f})')

# ── KS & ROC plots ────────────────────────────────────────────────────────────
scores_0 = np.sort(y_test_proba[y_test == 0])
scores_1 = np.sort(y_test_proba[y_test == 1])
cdf_0    = np.arange(1, len(scores_0)+1) / len(scores_0)
cdf_1    = np.arange(1, len(scores_1)+1) / len(scores_1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=100)

axes[0].plot(scores_0, cdf_0, color='#1a434e', lw=2, label='Negative class')
axes[0].plot(scores_1, cdf_1, color='#e74c3c', lw=2, label='Positive class')
axes[0].axvline(x=ks_thresh, color='#8e44ad', ls='--', lw=1.5, label=f'KS = {ks_stat:.4f}')
axes[0].set(xlabel='Predicted Probability', ylabel='Cumulative Proportion', title='KS Statistic')
axes[0].legend(fontsize=9)

axes[1].plot(fpr_ks, tpr_ks, color='#1a434e', lw=2,
             label=f'AUC={test_auc:.4f}, Gini={gini:.4f}')
axes[1].fill_between(fpr_ks, tpr_ks, alpha=0.08, color='#1a434e')
axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve with Gini')
axes[1].legend(fontsize=9)

sns.despine(); plt.tight_layout(); plt.show()


## 3️⃣  Gains & Lift Table

In [ ]:
gains_df = pd.DataFrame({'score': y_test_proba, 'actual': y_test.values})
gains_df = gains_df.sort_values('score', ascending=False).reset_index(drop=True)

n_total    = len(gains_df)
n_positives = gains_df['actual'].sum()
n_deciles  = 10

gains_df['decile'] = pd.qcut(gains_df.index, q=n_deciles, labels=range(1, n_deciles+1))

gains_table = (
    gains_df.groupby('decile', observed=True)
    .agg(n_accounts=('actual','count'), n_positives=('actual','sum'), avg_score=('score','mean'))
    .reset_index()
)
gains_table['cum_accounts']     = gains_table['n_accounts'].cumsum()
gains_table['cum_positives']    = gains_table['n_positives'].cumsum()
gains_table['cum_pct_accounts'] = gains_table['cum_accounts'] / n_total * 100
gains_table['cum_pct_captured'] = gains_table['cum_positives'] / n_positives * 100
gains_table['lift']             = (gains_table['n_positives'] / gains_table['n_accounts']) / (n_positives / n_total)

top_decile_capture = gains_table.iloc[0]['cum_pct_captured']
print(f'Top-decile capture: {top_decile_capture:.1f}%')
print()
print(gains_table.to_string(index=False, float_format='{:.2f}'.format))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=100)
axes[0].plot(gains_table['cum_pct_accounts'], gains_table['cum_pct_captured'],
             color='#1a434e', lw=2.5, marker='o', ms=5, label=CHAMPION_NAME)
axes[0].plot([0,100],[0,100],'k--',lw=1,alpha=0.4,label='Random')
axes[0].axhline(y=top_decile_capture, color='#e74c3c', ls=':', lw=1,
                label=f'Top decile: {top_decile_capture:.1f}%')
axes[0].set(xlabel='% Accounts Reviewed', ylabel='% Positives Captured', title='Cumulative Gains')
axes[0].legend(fontsize=9)

axes[1].bar(gains_table['decile'].astype(str), gains_table['lift'],
            color='#1a434e', edgecolor='white')
axes[1].axhline(y=1.0, color='#e74c3c', ls='--', lw=1.5, label='Random (lift=1)')
axes[1].set(xlabel='Score Decile (1=Highest Risk)', ylabel='Lift', title='Lift by Decile')
axes[1].legend(fontsize=9)

sns.despine(); plt.tight_layout(); plt.show()


## 4️⃣  Calibration (Brier Score)

In [ ]:
fraction_pos, mean_pred = calibration_curve(y_test, y_test_proba, n_bins=10, strategy='uniform')
brier          = brier_score_loss(y_test, y_test_proba)
brier_baseline = y_test.mean() * (1 - y_test.mean())

print(f'Brier Score          : {brier:.4f}')
print(f'No-skill Brier       : {brier_baseline:.4f}')
print(f'Brier Skill Score    : {1 - brier / brier_baseline:.4f}  (1 = perfect, 0 = no skill)')

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
ax.plot(mean_pred, fraction_pos, color='#1a434e', lw=2.5, marker='o', ms=6,
        label=f'{CHAMPION_NAME} (Brier={brier:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Perfect calibration')
ax.set(xlabel='Mean Predicted Probability', ylabel='Fraction of Positives', title='Calibration Curve')
ax.legend(fontsize=10)
sns.despine(); plt.tight_layout(); plt.show()


## 5️⃣  All-Model Comparison on Test Set

In [ ]:
test_results = []
for name, entry in model_registry.items():
    test_proba = entry['pipeline'].predict_proba(X_test)[:, 1]
    t_auc      = roc_auc_score(y_test, test_proba)
    t_gini     = 2 * t_auc - 1
    fpr_t, tpr_t, _ = roc_curve(y_test, test_proba)
    t_ks       = float(np.max(tpr_t - fpr_t))
    test_results.append({
        'Model'    : name,
        'Val AUC'  : round(entry['val_auc'], 4),
        'Test AUC' : round(t_auc,  4),
        'AUC Delta': round(t_auc - entry['val_auc'], 4),
        'Gini'     : round(t_gini, 4),
        'KS'       : round(t_ks,   4),
    })
    entry['test_proba'] = test_proba

results_df = (pd.DataFrame(test_results).sort_values('Test AUC', ascending=False).reset_index(drop=True))
results_df.index += 1
print('='*70)
print('FULL MODEL COMPARISON — Test Set')
print('='*70)
print(results_df.to_string())
print()
print('AUC Delta = Test AUC − Val AUC  (negative = val was optimistic)')


## 6️⃣  Success Criteria Scorecard

**CHANGE:** Update thresholds to match your project's acceptance criteria.

In [ ]:
# CHANGE: set your project's success thresholds and metrics
criteria = {
    'AUC-ROC'   : (test_auc,  0.78, '>'),   # CHANGE threshold
    'Gini'      : (gini,      0.55, '>'),   # CHANGE threshold
    'KS'        : (ks_stat,   0.35, '>'),   # CHANGE threshold
    # Add more criteria as needed
}

print('='*65)
print(f'SUCCESS CRITERIA SCORECARD — {CHAMPION_NAME}')
print('='*65)
print(f'  {"Metric":<30}  {"Achieved":>9}  {"Threshold":>10}  {"Status":>8}')
print('  ' + '-'*60)

all_pass = True
for metric, (value, threshold, direction) in criteria.items():
    passed = (value > threshold) if direction == '>' else (value < threshold)
    status = 'PASS' if passed else 'FAIL'
    if not passed: all_pass = False
    print(f'  {metric:<30}  {value:>9.4f}  {threshold:>10.2f}  {status:>8}')

print('  ' + '-'*60)
print()
if all_pass:
    print('  ✅  ALL CRITERIA MET')
else:
    print('  ❌  SOME CRITERIA NOT MET — review tuning or feature engineering.')
